<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 2 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Observe Storage and Write Batches</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Compare two ways of writing the same orders to understand data results and storage state.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Dedicated lab database</span>
</div>

By the end, you will compare batch and row-by-row writes of the same orders, verify that both tables contain ten rows totaling 12220.60, and observe Tablet metadata. Run the cells in order.

[Course notes](course2_doris_architecture.md) · [Course home](../README.md)


## Lab Scope

Rebuild only orders_batch and orders_rowwise. Use the same ten orders to compare writing ten rows at once with writing them in ten separate requests. Background Compaction continually reorganizes data, so record the sampling time when observing metadata.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. Keep Variables Consistent

Both tables use the same fields, ten orders, one bucket, and one replica. Set this session's Group Commit to off_mode to observe each write: orders_batch submits ten rows at once, while orders_rowwise submits one row at a time.

Each write batch incurs commit and version-management overhead; this step uses identical business data to observe the effect of batching.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_batch")
ddl = order_ddl("orders_batch")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rowwise")
ddl = order_ddl("orders_rowwise")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
rows = history_rows()
lab.execute("SET group_commit = 'off_mode'")
lab.insert("orders_batch", ORDER_COLUMNS, rows)
for row in rows:
    lab.insert("orders_rowwise", ORDER_COLUMNS, [row])


## 2. Query Metadata

SHOW TABLETS lists a table's Tablets and version-related information. First confirm that each table has one Tablet, then compare VersionCount. It reflects storage version state at sampling time; if background compaction has already run, the difference between the tables may be smaller.


In [ ]:
lab.sql("SHOW CREATE TABLE orders_batch");
lab.sql("SHOW PARTITIONS FROM orders_batch");
lab.sql("SHOW TABLETS FROM orders_batch");
lab.sql("SHOW TABLETS FROM orders_rowwise");


## 3. First Confirm That Business Results Are Unchanged

Both tables should contain 10 rows with a pre-tax total of 12220.60. Check details and summaries before interpreting metadata: changing write batches can change physical organization, but business orders and amounts should remain identical.


In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "12220.60")])
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1");
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_batch", title="Batch write results")
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_rowwise", title="Row-by-row write results")


## Limits of This Lab's Observations

This lab observes business results separately from storage state: row counts and amounts check data completeness, while Tablet metadata helps explain writes and compaction. The ten-row sample illustrates the mechanism; production performance requires larger data volumes, fixed concurrency, and sustained sampling.


## Your turn

Run EXPLAIN on each table reading only order_id and order_amount, then add other columns and identify the changes in the plan's output columns. EXPLAIN shows the plan; to see actual scan volume, execute the query and inspect Query Profile.

Sample VersionCount again and interpret the results in light of the sampling interval.


## Independent exercise

Query the first day's order count and amount from each of the two write-method tables, and compare them in one result table. Expect 5 orders and 3944.20 for each. Then inspect EXPLAIN for the corresponding filter and identify the date filter condition.

Write and run your code in the next cell, then expand the reference solution after finishing. A blank exercise is not automatically marked as complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
query = """SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_batch WHERE order_date = '2013-01-01'
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount)
FROM orders_rowwise WHERE order_date = '2013-01-01'
ORDER BY write_mode"""
lab.sql(query, title="First-day orders from both write methods")
expect(lab.query(query), [("batch", 5, "3944.20"), ("rowwise", 5, "3944.20")])
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_date = '2013-01-01'", title="Filter plan")
```

</details>
